# ENDISI Calculation

Enhanced Normalized Difference Impervious Surfaces Index from Sentinel-2 via MPC

## Prepare Libraries

In [ ]:
# Load source data from a local or configured project directory.

In [ ]:
# # Install required libraries
# !pip install pystac_client planetary_computer rasterio rioxarray scikit-image -q

In [ ]:
# Import Essential Libraries
from pystac.extensions.eo import EOExtension as eo
from PIL import Image
import pystac_client
import planetary_computer as pc
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import os

import rasterio
from rasterio import features, windows, warp
from rasterio.enums import Resampling

import rioxarray
import xarray as xr
import tensorflow as tf
from skimage.filters import threshold_otsu


In [ ]:
# Check for available GPU
physical_devices = tf.config.list_physical_devices('GPU')
if physical_devices:
    print(f'GPU found => {physical_devices}')
    # Set memory growth to avoid memory allocation issues
    for device in physical_devices:
        tf.config.experimental.set_memory_growth(device, True)
else:
    print('No GPU found.')

## Access Satellite Images

In [ ]:
# Data access
catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=pc.sign_inplace,
)

print("Connected to MPC Catalog")

In [ ]:
# Call the AOI of Jakarta
# import rasterio
# from rasterio.mask import mask
# import geopandas as gpd

# # Define file paths
# shapefile_path = r"[R2402] British Academy - Empowering Resilience In a Sinking City\2. Data\1. Big Data\Area of Interest (Source_ Jakarta Satu)\City\adm_dki-jakarta.shp"

# # Read shapefile
# aoi = gpd.read_file(shapefile_path)
# aoi = aoi.to_crs("EPSG:4326")  # Ensure CRS matches raster

# # Plot the Area of Interest (AOI)
# aoi.plot(figsize=(8, 6), edgecolor='black', legend=False)

In [ ]:
# Define Area of Interest (AOI) of Gelora Bung Karno, South Jakarta
# list the coordinates
left = 106.8
right = 106.85
top = -6.15
bottom = -6.2

aoi = {
    "type": "Polygon",
    "coordinates": [
        [
            [left, top],
            [right, top],
            [right, bottom],
            [left, bottom],
            [left, top],  # Closing the polygon
        ]
    ],
}

In [ ]:
# Define Date Range
date = "2024-01-01/2024-12-31"

In [ ]:
# Define Could Cover tolerance
cc = 10

In [ ]:
# Search for Sentinel-2 data with cloud cover filter
search = catalog.search(
    collections=["sentinel-2-l2a"],  # Dataset
    intersects=aoi,    # Area of Interest
    datetime=date,  # Date range
    query={"eo:cloud_cover": {"lt": cc}},  # Cloud cover filter (<10%)
)

In [ ]:
# Check how many items were found
items = list(search.items())
print(f"Found {len(items)} items with cloud cover < {cc}%:")
for i in items:
    print(f"<> {i.datetime.date()} | cloud cover {eo.ext(i).cloud_cover}% | {i.id}")

In [ ]:
# Choose the least cloudy item
least_cloudy_item = min(items, key=lambda item: eo.ext(item).cloud_cover)

print(
    f"Choosing {least_cloudy_item.id} from {least_cloudy_item.datetime.date()}"
    f" with {eo.ext(least_cloudy_item).cloud_cover}% cloud cover."
)

## Visualise the AOI

In [ ]:
# Visualise the Area of Interest (AOI)
# Assuming 'least_cloudy_item', 'aoi', and 'band_data' are already defined
with rasterio.open(least_cloudy_item.assets["visual"].href) as ds:
    # Get image bounds in dataset's CRS
    aoi_bounds = features.bounds(aoi)
    warped_aoi_bounds = warp.transform_bounds("epsg:4326", ds.crs, *aoi_bounds)
    aoi_window = windows.from_bounds(transform=ds.transform, *warped_aoi_bounds)

    # Read band data for the AOI
    band_data = ds.read(window=aoi_window)

# Create image and resize
img = Image.fromarray(np.transpose(band_data, axes=[1, 2, 0]))
w, h = img.size
aspect = w / h
target_w = 1000
target_h = int(target_w / aspect)
img = img.resize((target_w, target_h), Image.Resampling.BILINEAR)

# Get dataset bounds in EPSG:4326 for coordinate grid
aoi_bounds = ds.window_bounds(aoi_window)  # Get bounds from the dataset window
left, bottom, right, top = warp.transform_bounds(ds.crs, "epsg:4326", *aoi_bounds)  # Get bounds in EPSG:4326

# Plotting with coordinate axes
fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(img)

# Set coordinate grid and labels
lon_ticks = np.linspace(left, right, 5)
lat_ticks = np.linspace(bottom, top, 5)

x_coords = np.linspace(0, img.size[0], len(lon_ticks))
y_coords = np.linspace(0, img.size[1], len(lat_ticks))

ax.xaxis.set_major_locator(plt.FixedLocator(x_coords))
ax.xaxis.set_major_formatter(plt.FixedFormatter([f"{lon:.4f}" for lon in lon_ticks]))

ax.yaxis.set_major_locator(plt.FixedLocator(y_coords))
ax.yaxis.set_major_formatter(plt.FixedFormatter([f"{lat:.4f}" for lat in lat_ticks]))

ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
plt.title("Sentinel-2-L2A Image of the AOI")
plt.show()

# Print the image size
print(f"AOI Image size: {target_w} (width) x {target_h} (height)")

In [ ]:
# Save the AOI image with coordinates to a GeoTIFF file
# Define base_path if not already defined
base_path = r'Impervious vs Pervious Surface\data'
os.makedirs(base_path, exist_ok=True)

# Create date_str from least_cloudy_item
date_str = least_cloudy_item.datetime.date().strftime("%Y-%m-%d")

# Convert the image to a numpy array (ensure it's in the correct shape: (height, width, bands))
aoi_img_array = np.array(img)

# If the image is RGB, transpose to (bands, height, width) for rasterio
if aoi_img_array.ndim == 3:
    aoi_img_array = np.transpose(aoi_img_array, (2, 0, 1))

# Define output path for GeoTIFF
aoi_geotiff_path = os.path.join(base_path, f"aoi_image_{date_str}.tif")

from rasterio.transform import from_bounds

# Compute transform for the resized AOI image
# Use left, bottom, right, top from the AOI bounds in EPSG:4326
# and the image size (target_w, target_h)
transform_img = from_bounds(left, bottom, right, top, target_w, target_h)

# Save as GeoTIFF with georeferencing
with rasterio.open(
    aoi_geotiff_path,
    "w",
    driver="GTiff",
    height=target_h,
    width=target_w,
    count=aoi_img_array.shape[0],
    dtype=aoi_img_array.dtype,
    crs="EPSG:4326",
    transform=transform_img,
) as dst:
    dst.write(aoi_img_array)

print(f"AOI image with coordinates saved to {aoi_geotiff_path}")

## Assign Images

In [ ]:
# Sign the item's assets for direct access
signed_item = pc.sign(least_cloudy_item)

# Collect band information
band_info = []

for asset_key, asset in signed_item.assets.items():
    bands = asset.extra_fields.get('eo:bands', [])
    for band in bands:
        band_info.append({
            "Asset Title": asset.title,
            #"Asset": asset_key,
            "Band Name": band.get("name", "N/A"),
            "Common Name": band.get("common_name", "N/A"),
            "Resolution (m)": asset.extra_fields.get("gsd", "N/A"),  # Extract spatial resolution
            "Center Wavelength (µm)": band.get("center_wavelength", "N/A"),
            "Full Width Half Max (µm)": band.get("full_width_half_max", "N/A"),
        })

In [ ]:
# Define the Sentinel-2 bands to use for the composite
band_names = [
    "B02", # Blue
    "B03", # Green
    "B04", # Red
    "B08", # NIR
    "B11", # SWIR 1
    "B12", # SWIR 2
]

In [ ]:
# Find the matching assets from the signed item
band_assets = {}
for band in band_names:
    asset = next(
        (asset for asset_key, asset in signed_item.assets.items()
         if asset.extra_fields.get("eo:bands", [{}])[0].get("name") == band),
        None,
    )
    if asset:
        band_assets[band] = asset.href

# Ensure we have all the bands
if len(band_assets) != len(band_names):
    missing_bands = set(band_names) - set(band_assets.keys())
    raise ValueError(f"Missing bands: {missing_bands}")

In [ ]:
# Assign Band 02 AOI as a single layer
with rasterio.open(band_assets["B02"]) as blue_src:
    # Transform AOI bounds to the dataset's CRS
    aoi_bounds = features.bounds(aoi)
    aoi_pixel_bounds = windows.from_bounds(
        *warp.transform_bounds("epsg:4326", blue_src.crs, *aoi_bounds),
        transform=blue_src.transform
    )

    # Read the data within the AOI
    aoi_window = windows.Window(
        col_off=int(aoi_pixel_bounds.col_off),
        row_off=int(aoi_pixel_bounds.row_off),
        width=int(aoi_pixel_bounds.width),
        height=int(aoi_pixel_bounds.height),
    )

    band2_aoi = blue_src.read(1, window=aoi_window)

    # Get dataset bounds in EPSG:4326 for the grid (for the cropped AOI)
    aoi_bounds = blue_src.window_bounds(aoi_window)
    lon_min, lat_min, lon_max, lat_max = warp.transform_bounds(blue_src.crs, "epsg:4326", *aoi_bounds)

print(band2_aoi.shape)

In [ ]:
# Assign Band 03 AOI as a single layer
with rasterio.open(band_assets["B03"]) as green_src:
    # Transform AOI bounds to the dataset's CRS
    aoi_bounds = features.bounds(aoi)
    aoi_pixel_bounds = windows.from_bounds(
        *warp.transform_bounds("epsg:4326", green_src.crs, *aoi_bounds),
        transform=green_src.transform
    )

    # Read the data within the AOI
    aoi_window = windows.Window(
        col_off=int(aoi_pixel_bounds.col_off),
        row_off=int(aoi_pixel_bounds.row_off),
        width=int(aoi_pixel_bounds.width),
        height=int(aoi_pixel_bounds.height),
    )

    band3_aoi = green_src.read(1, window=aoi_window)

    # Get dataset bounds in EPSG:4326 for the grid (for the cropped AOI)
    aoi_bounds = green_src.window_bounds(aoi_window)
    lon_min, lat_min, lon_max, lat_max = warp.transform_bounds(green_src.crs, "epsg:4326", *aoi_bounds)

print(band3_aoi.shape)

In [ ]:
# Assign Band 04 AOI as a single layer
with rasterio.open(band_assets["B04"]) as red_src:
    # Transform AOI bounds to the dataset's CRS
    aoi_bounds = features.bounds(aoi)
    aoi_pixel_bounds = windows.from_bounds(
        *warp.transform_bounds("epsg:4326", red_src.crs, *aoi_bounds),
        transform=red_src.transform
    )

    # Read the data within the AOI
    aoi_window = windows.Window(
        col_off=int(aoi_pixel_bounds.col_off),
        row_off=int(aoi_pixel_bounds.row_off),
        width=int(aoi_pixel_bounds.width),
        height=int(aoi_pixel_bounds.height),
    )

    band4_aoi = red_src.read(1, window=aoi_window)

    # Get dataset bounds in EPSG:4326 for the grid (for the cropped AOI)
    aoi_bounds = red_src.window_bounds(aoi_window)
    lon_min, lat_min, lon_max, lat_max = warp.transform_bounds(red_src.crs, "epsg:4326", *aoi_bounds)

print(band4_aoi.shape)

In [ ]:
# Assign Band 08 AOI as a single layer
with rasterio.open(band_assets["B08"]) as nir_src:
    # Transform AOI bounds to the dataset's CRS
    aoi_bounds = features.bounds(aoi)
    aoi_pixel_bounds = windows.from_bounds(
        *warp.transform_bounds("epsg:4326", nir_src.crs, *aoi_bounds),
        transform=nir_src.transform
    )

    # Read the data within the AOI
    aoi_window = windows.Window(
        col_off=int(aoi_pixel_bounds.col_off),
        row_off=int(aoi_pixel_bounds.row_off),
        width=int(aoi_pixel_bounds.width),
        height=int(aoi_pixel_bounds.height),
    )

    band8_aoi = nir_src.read(1, window=aoi_window)

    # Get dataset bounds in EPSG:4326 for the grid (for the cropped AOI)
    aoi_bounds = nir_src.window_bounds(aoi_window)
    lon_min, lat_min, lon_max, lat_max = warp.transform_bounds(nir_src.crs, "epsg:4326", *aoi_bounds)

print(band8_aoi.shape)

In [ ]:
# Assign Band 11 AOI as a single layer and Resample into the same resolution
with rasterio.open(band_assets["B11"]) as swir1_src:
    # Transform AOI bounds to the dataset's CRS
    aoi_bounds = features.bounds(aoi)
    aoi_pixel_bounds = windows.from_bounds(
        *warp.transform_bounds("epsg:4326", swir1_src.crs, *aoi_bounds),
        transform=swir1_src.transform
    )

    # Read the data within the AOI
    aoi_window = windows.Window(
        col_off=int(aoi_pixel_bounds.col_off),
        row_off=int(aoi_pixel_bounds.row_off),
        width=int(aoi_pixel_bounds.width),
        height=int(aoi_pixel_bounds.height),
    )

    band11_aoi = swir1_src.read(1, window=aoi_window)

    # Get dataset bounds in EPSG:4326 for the grid (for the cropped AOI)
    aoi_bounds = swir1_src.window_bounds(aoi_window)
    lon_min, lat_min, lon_max, lat_max = warp.transform_bounds(swir1_src.crs, "epsg:4326", *aoi_bounds)

print(band11_aoi.shape)

# Image Resampling for Band 11
# Create an in-memory dataset from band11_aoi
with rasterio.io.MemoryFile() as memfile:
    with memfile.open(
        driver='GTiff',
        height=band11_aoi.shape[0],
        width=band11_aoi.shape[1],
        count=1,  # Single band
        dtype=band11_aoi.dtype,
        crs=swir1_src.crs,  # Use the CRS from the original source
        transform=swir1_src.transform,  # Use the transform from the original source
    ) as dataset:
        dataset.write(band11_aoi, 1)  # Write the band data

        # Set upscale factor (20m to 10m => factor of 2)
        upscale_factor = 2

        # Calculate new height and width using the upscale factor
        new_width = dataset.width * upscale_factor
        new_height = dataset.height * upscale_factor

        # Generate new transform
        new_transform = dataset.transform * dataset.transform.scale(
            1 / upscale_factor, 1 / upscale_factor
        )

        # Create a new array to hold the resampled data
        band11_aoi = dataset.read(
            out_shape=(1, int(new_height), int(new_width)),
            resampling=Resampling.bilinear,  # Choose the resampling method
        )
        band11_aoi = np.squeeze(band11_aoi)

print(band11_aoi.shape)

In [ ]:
# Assign Band 12 AOI as a single layer and Resample into the same resolution
with rasterio.open(band_assets["B12"]) as swir2_src:
    # Transform AOI bounds to the dataset's CRS
    aoi_bounds = features.bounds(aoi)
    aoi_pixel_bounds = windows.from_bounds(
        *warp.transform_bounds("epsg:4326", swir2_src.crs, *aoi_bounds),
        transform=swir2_src.transform
    )

    # Read the data within the AOI
    aoi_window = windows.Window(
        col_off=int(aoi_pixel_bounds.col_off),
        row_off=int(aoi_pixel_bounds.row_off),
        width=int(aoi_pixel_bounds.width),
        height=int(aoi_pixel_bounds.height),
    )

    band12_aoi = swir2_src.read(1, window=aoi_window)

    # Get dataset bounds in EPSG:4326 for the grid (for the cropped AOI)
    aoi_bounds = swir2_src.window_bounds(aoi_window)
    lon_min, lat_min, lon_max, lat_max = warp.transform_bounds(swir2_src.crs, "epsg:4326", *aoi_bounds)

print(band12_aoi.shape)

# Image Resampling for Band 12
# Create an in-memory dataset from band12_aoi
with rasterio.io.MemoryFile() as memfile:
    with memfile.open(
        driver='GTiff',
        height=band12_aoi.shape[0],
        width=band12_aoi.shape[1],
        count=1,  # Single band
        dtype=band12_aoi.dtype,
        crs=swir2_src.crs,  # Use the CRS from the original source
        transform=swir2_src.transform,  # Use the transform from the original source
    ) as dataset:
        dataset.write(band12_aoi, 1)  # Write the band data

        # Set upscale factor (20m to 10m => factor of 2)
        upscale_factor = 2

        # Calculate new height and width using the upscale factor
        new_width = dataset.width * upscale_factor
        new_height = dataset.height * upscale_factor

        # Generate new transform
        new_transform = dataset.transform * dataset.transform.scale(
            1 / upscale_factor, 1 / upscale_factor
        )

        # Create a new array to hold the resampled data
        band12_aoi = dataset.read(
            out_shape=(1, int(new_height), int(new_width)),
            resampling=Resampling.bilinear,  # Choose the resampling method
        )
        band12_aoi = np.squeeze(band12_aoi)

print(band12_aoi.shape)

In [ ]:
# Stack band variables (ensure all bands are the same shape)
bands = [
    band2_aoi,
    band3_aoi,
    # band4_aoi,
    # band8_aoi,
    band11_aoi,
    band12_aoi
]

# Stack into a 3D numpy array (bands, height, width)
stacked_array = np.stack(bands, axis=0)

# Use the same georeferencing as the AOI image
stacked_geotiff_path = os.path.join(base_path, f"stacked_aoi_{date_str}.tif")

with rasterio.open(
    stacked_geotiff_path,
    "w",
    driver="GTiff",
    height=target_h,
    width=target_w,
    count=stacked_array.shape[0],
    dtype=stacked_array.dtype,
    crs="EPSG:4326",
    transform=transform_img,
) as dst:
    dst.write(stacked_array)

print(f"Stacked GeoTIFF saved at {stacked_geotiff_path}")


## Spectral Transformation

In [ ]:
# Create a mapping of band names to their AOI arrays
band_map = {
	"B02": band2_aoi,
	"B03": band3_aoi,
	# "B04": band4_aoi, # not used for ENDISI calculation
	# "B08": band8_aoi, # not used for ENDISI calculation
	"B11": band11_aoi,
	"B12": band12_aoi,
}

# Get shape and georeferencing from one of the bands (all should match)
height, width = band2_aoi.shape
transform = transform_img  # already in EPSG:4326
crs = "EPSG:4326"

# Build xarray Dataset from numpy arrays
coords = {
	"y": np.linspace(top, bottom, height),
	"x": np.linspace(left, right, width),
}
da = xr.Dataset({
	name: xr.DataArray(
		data=arr,
		dims=("y", "x"),
		coords=coords,
		attrs={"transform": transform, "crs": crs}
	)
	for name, arr in band_map.items()
})

# Assign spatial reference for plotting and area computation
for var in da.data_vars:
	da[var].rio.set_spatial_dims(x_dim="x", y_dim="y", inplace=True)
	da[var].rio.write_crs(crs, inplace=True)

### Calculate MNDWI

In [ ]:
# Calculate MNDWI (Modified Normalized Difference Water Index)
# Using Green (B03) and SWIR 1 (B11) for the ratio
green = da["B03"]
swir1 = da["B11"]
mndwi = (green - swir1) / (green + swir1)

### Compute β (beta) Correction Factor

In [ ]:
# Compute β (beta) Correction Factor
# Using Blue (B02) and SWIR 1 (B11) for the ratio
blue = da["B02"]
swir2 = da["B12"]

swir_ratio = swir1 / swir2
mndwi_sq = mndwi ** 2

beta = (2 * blue.mean()) / (swir_ratio.mean() + mndwi_sq.mean())
print(f"β (beta) correction factor: {beta.values:.4f}")


### Compute ENDISI

In [ ]:
# Compute ENDISI (Enhanced Normalized Difference Impervious Surface Index)
# Using the formula: ENDISI = (Blue - β * (SWIR Ratio + M
numerator = blue - beta * (swir_ratio + mndwi_sq)
denominator = blue + beta * (swir_ratio + mndwi_sq)
endisi = numerator / denominator

# Clip to -1 to 1 to avoid extreme outliers
endisi = endisi.clip(-1, 1)

## Visualise ENDISI

In [ ]:
# Visualise ENDISI
plt.figure(figsize=(8, 8))
endisi.plot(cmap="RdYlGn_r", vmin=-1, vmax=1)
plt.title("ENDISI - Enhanced Normalized Difference Impervious Surfaces Index")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.show()

In [ ]:
# Save the ENDISI result as a GeoTIFF
endisitiff_path = os.path.join(base_path, f"impervious_surface_endisi_{date_str}.tif")

with rasterio.open(
    endisitiff_path,
    "w",
    driver="GTiff",
    height=target_h,
    width=target_w,
    count=1,
    dtype=endisi.values.dtype,
    crs="EPSG:4326",
    transform=transform_img,
) as dst:
    dst.write(endisi.values, 1)

print(f"ENDISI result saved to {endisitiff_path}")


## Reclassify Impervious Surfaces with Threshold

### Otsu-based Threshold

In [ ]:
# Reclassify Impervious Surfaces with Threshold using Otsu-based threshold
# Flatten the ENDISI data and remove NaN values for thresholding
endisi_data = endisi.where(~np.isnan(endisi)).values.flatten()

# Otsu's threshold (automatic)
otsu_thresh = threshold_otsu(endisi_data[~np.isnan(endisi_data)])

# Create impervious areas mask
impervious_mask_otsu = endisi > otsu_thresh

# Plot the reclassified ENDISI
plt.figure(figsize=(8, 8))
impervious_mask_otsu.plot(
    cmap="gray", vmin=0, vmax=1,
    x="x", y="y"
)
plt.title(f"Urban Mask (threshold ≈ {otsu_thresh:.3f})")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.show()


In [ ]:
# Save the impervious_mask result as a GeoTIFF using the same CRS and transform as above
impervious_mask_otsu_path = os.path.join(base_path, f"impervious_surface_binary_otsu_{date_str}.tif")

with rasterio.open(
    impervious_mask_otsu_path,
    "w",
    driver="GTiff",
    height=target_h,
    width=target_w,
    count=1,
    dtype="uint8",
    crs="EPSG:4326",
    transform=transform_img,
) as dst:
    dst.write(impervious_mask_otsu.astype("uint8").values, 1)

print(f"Impervious mask saved to {impervious_mask_otsu_path}")

### Manual Threshold

In [ ]:
# Reclassify Impervious Surfaces using manual threshold
# Define the threshold
manual_thresh = -0.45

# Create a binary mask where ENDISI > threshold
impervious_mask_manual = (endisi > manual_thresh).astype("uint8")

# Plot the reclassified ENDISI
plt.figure(figsize=(8, 8))
impervious_mask_manual.plot(
    cmap="gray", vmin=0, vmax=1,
    x="x", y="y"
)
plt.title("Binary ENDISI (threshold = -0.45)")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.show()

In [ ]:
# Save the impervious_mask result (from manual threshold) as a GeoTIFF using the same CRS and transform as above
impervious_mask_manual_tiff_path = os.path.join(base_path, f"impervious_surface_binary_manual_{date_str}.tif")

with rasterio.open(
    impervious_mask_manual_tiff_path,
    "w",
    driver="GTiff",
    height=target_h,
    width=target_w,
    count=1,
    dtype="uint8",
    crs="EPSG:4326",
    transform=transform_img,
) as dst:
    dst.write(impervious_mask_manual.astype("uint8").values, 1)

print(f"Impervious mask (manual threshold={manual_thresh}) saved to {impervious_mask_manual_tiff_path}")